# Lab 4 – Task 2 & Task 3: Problem-Solving Planning Agents

This notebook contains the graded planning-agent tasks:
- **Task 2:** Cloud Resource Deployment Planning Agent
- **Task 3:** University Course Registration Planning Agent

# Task 2: Cloud Resource Deployment Planning Agent

A Problem-Solving Agent that plans the deployment of an AI application onto cloud infrastructure using uniform-cost search over a state-space graph.

In [ ]:
from collections import deque
import heapq    

## 1. Problem Formulation

- **Initial State:** `S0` — Deployment request received
- **Goal State:** `S6` — Deployment Completed
- **Actions:** Allocate Resources, Create Virtual Machine, Install Dependencies, Deploy AI Model, Verify Deployment
- **Transition Model:** Each completed action moves the system to the next deployment state
- **Goal Test:** Deployment status equals Completed
- **Path Cost:** Execution time (minutes) of each action

In [2]:
INITIAL_STATE = "S0"          # Deployment request received
GOAL_STATE = "S6"             # Deployment Completed (Goal)

STATE_DESCRIPTIONS = {
    "S0": "Request Received",
    "S1": "Allocate Resources",
    "S2": "Create VM",
    "S3": "Install Dependencies",
    "S4": "Deploy AI Model",
    "S5": "Verify Deployment",
    "S6": "Deployment Completed (Goal)",
}

# Transition Model: state -> list of (action, next_state, cost/time in minutes)
ACTIONS = {
    "S0": [("Allocate Resources", "S1", 2)],
    "S1": [("Create Virtual Machine", "S2", 4)],
    "S2": [("Install Dependencies", "S3", 3)],
    "S3": [("Deploy AI Model", "S4", 5)],
    "S4": [("Verify Deployment", "S5", 2)],
    "S5": [("Confirm Deployment", "S6", 1)],
    "S6": [],
}


def goal_test(state):
    return state == GOAL_STATE

## 2. Generate the Complete Deployment State Space

Breadth-first traversal that discovers every reachable state and every transition edge.

In [3]:
def generate_state_space(initial_state):
    visited = set()
    queue = deque([initial_state])
    edges = []

    while queue:
        state = queue.popleft()
        if state in visited:
            continue
        visited.add(state)

        for action, next_state, cost in ACTIONS.get(state, []):
            edges.append((state, action, next_state, cost))
            if next_state not in visited:
                queue.append(next_state)

    return visited, edges

## 3. Uniform-Cost Search

Finds the minimum-cost deployment sequence from the initial state to the goal state.

In [4]:
def uniform_cost_search(initial_state):
    frontier = [(0, initial_state, [])]   # (cumulative_cost, state, path_of_actions)
    visited = set()

    while frontier:
        cost_so_far, state, path = heapq.heappop(frontier)

        if goal_test(state):
            return path, cost_so_far

        if state in visited:
            continue
        visited.add(state)

        for action, next_state, step_cost in ACTIONS.get(state, []):
            if next_state not in visited:
                heapq.heappush(
                    frontier,
                    (cost_so_far + step_cost, next_state, path + [(state, action, next_state, step_cost)])
                )

    return None, None

## 4. Run the Agent

In [5]:
print("=" * 60)
print("CLOUD RESOURCE DEPLOYMENT PLANNING AGENT")
print("=" * 60)

print(f"\nInitial State : {INITIAL_STATE} - {STATE_DESCRIPTIONS[INITIAL_STATE]}")
print(f"Goal State    : {GOAL_STATE} - {STATE_DESCRIPTIONS[GOAL_STATE]}")

states, edges = generate_state_space(INITIAL_STATE)

print("\n--- Complete Deployment State Space ---")
print(f"States ({len(states)}): {sorted(states)}")
print("Transitions:")
for state, action, next_state, cost in edges:
    print(f"  {state} --[{action} ({cost} min)]--> {next_state}")

plan, total_cost = uniform_cost_search(INITIAL_STATE)

print("\n--- Planned Execution Sequence ---")
if plan is None:
    print("No valid plan found to reach the goal state.")
else:
    sequence = [INITIAL_STATE] + [step[2] for step in plan]
    print(" -> ".join(sequence))

    print("\n--- Action Sequence ---")
    for state, action, next_state, cost in plan:
        print(f"  {state} -> {next_state} : {action} ({cost} min)")

    print(f"\nTotal Deployment Time = "
          f"{' + '.join(str(step[3]) for step in plan)} = {total_cost} min")

CLOUD RESOURCE DEPLOYMENT PLANNING AGENT

Initial State : S0 - Request Received
Goal State    : S6 - Deployment Completed (Goal)

--- Complete Deployment State Space ---
States (7): ['S0', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6']
Transitions:
  S0 --[Allocate Resources (2 min)]--> S1
  S1 --[Create Virtual Machine (4 min)]--> S2
  S2 --[Install Dependencies (3 min)]--> S3
  S3 --[Deploy AI Model (5 min)]--> S4
  S4 --[Verify Deployment (2 min)]--> S5
  S5 --[Confirm Deployment (1 min)]--> S6

--- Planned Execution Sequence ---
S0 -> S1 -> S2 -> S3 -> S4 -> S5 -> S6

--- Action Sequence ---
  S0 -> S1 : Allocate Resources (2 min)
  S1 -> S2 : Create Virtual Machine (4 min)
  S2 -> S3 : Install Dependencies (3 min)
  S3 -> S4 : Deploy AI Model (5 min)
  S4 -> S5 : Verify Deployment (2 min)
  S5 -> S6 : Confirm Deployment (1 min)

Total Deployment Time = 2 + 4 + 3 + 5 + 2 + 1 = 17 min


---

# Task 3: University Course Registration Planning Agent

A Problem-Solving Agent that determines the complete sequence of actions required for a student to successfully complete course registration, represented as a graph search that never revisits a completed step.

In [6]:
from collections import deque
import heapq

## 1. Problem Formulation

- **Initial State:** `S0` — Student Login
- **Goal State:** `S5` — Confirm Enrollment
- **Actions:** Authenticate Student, Verify Prerequisites, Select Courses, Verify Fee Status, Confirm Enrollment
- **Transition Model:** Each completed task updates the registration state
- **Goal Test:** Registration status equals Completed
- **Path Cost:** Each registration step has a processing cost

In [7]:
INITIAL_STATE = "S0"          # Student Login
GOAL_STATE = "S5"             # Confirm Enrollment (Goal)

STATE_DESCRIPTIONS = {
    "S0": "Student Login",
    "S1": "Authenticate Student",
    "S2": "Verify Prerequisites",
    "S3": "Select Courses",
    "S4": "Verify Fee Status",
    "S5": "Confirm Enrollment (Goal)",
}

# Transition Model: state -> list of (action, next_state, cost)
ACTIONS = {
    "S0": [("Authenticate Student", "S1", 1)],
    "S1": [("Verify Prerequisites", "S2", 2)],
    "S2": [("Select Courses", "S3", 2)],
    "S3": [("Verify Fee Status", "S4", 1)],
    "S4": [("Confirm Enrollment", "S5", 1)],
    "S5": [],
}


def goal_test(state):
    return state == GOAL_STATE

## 2. Generate All Reachable Registration States

BFS traversal that discovers every reachable state, never revisiting a completed step.

In [8]:
def generate_state_space(initial_state):
    visited = set()
    queue = deque([initial_state])
    edges = []

    while queue:
        state = queue.popleft()
        if state in visited:
            continue
        visited.add(state)

        for action, next_state, cost in ACTIONS.get(state, []):
            edges.append((state, action, next_state, cost))
            if next_state not in visited:
                queue.append(next_state)

    return visited, edges

## 3. Search for the Registration Plan

Uniform-cost search that finds the minimum-cost registration sequence, keeping a visited set so completed steps are never revisited.

In [9]:
def uniform_cost_search(initial_state):
    frontier = [(0, initial_state, [])]
    visited = set()   # completed registration steps are never revisited

    while frontier:
        cost_so_far, state, path = heapq.heappop(frontier)

        if goal_test(state):
            return path, cost_so_far

        if state in visited:
            continue
        visited.add(state)

        for action, next_state, step_cost in ACTIONS.get(state, []):
            if next_state not in visited:
                heapq.heappush(
                    frontier,
                    (cost_so_far + step_cost, next_state, path + [(state, action, next_state, step_cost)])
                )

    return None, None

## 4. Run the Agent

In [10]:
print("=" * 60)
print("UNIVERSITY COURSE REGISTRATION PLANNING AGENT")
print("=" * 60)

print(f"\nInitial State : {INITIAL_STATE} - {STATE_DESCRIPTIONS[INITIAL_STATE]}")
print(f"Goal State    : {GOAL_STATE} - {STATE_DESCRIPTIONS[GOAL_STATE]}")

states, edges = generate_state_space(INITIAL_STATE)

print("\n--- Reachable Registration State Space ---")
print(f"States ({len(states)}): {sorted(states)}")
print("Transitions:")
for state, action, next_state, cost in edges:
    print(f"  {state} --[{action} (cost {cost})]--> {next_state}")

plan, total_cost = uniform_cost_search(INITIAL_STATE)

print("\n--- Planned Execution Sequence ---")
if plan is None:
    print("No valid registration plan found.")
else:
    sequence = [INITIAL_STATE] + [step[2] for step in plan]
    print(" -> ".join(sequence))

    print("\n--- Action Sequence ---")
    for state, action, next_state, cost in plan:
        print(f"  {state} -> {next_state} : {action} (cost {cost})")

    print(f"\nTotal Registration Steps (Cost) = "
          f"{' + '.join(str(step[3]) for step in plan)} = {total_cost}")

UNIVERSITY COURSE REGISTRATION PLANNING AGENT

Initial State : S0 - Student Login
Goal State    : S5 - Confirm Enrollment (Goal)

--- Reachable Registration State Space ---
States (6): ['S0', 'S1', 'S2', 'S3', 'S4', 'S5']
Transitions:
  S0 --[Authenticate Student (cost 1)]--> S1
  S1 --[Verify Prerequisites (cost 2)]--> S2
  S2 --[Select Courses (cost 2)]--> S3
  S3 --[Verify Fee Status (cost 1)]--> S4
  S4 --[Confirm Enrollment (cost 1)]--> S5

--- Planned Execution Sequence ---
S0 -> S1 -> S2 -> S3 -> S4 -> S5

--- Action Sequence ---
  S0 -> S1 : Authenticate Student (cost 1)
  S1 -> S2 : Verify Prerequisites (cost 2)
  S2 -> S3 : Select Courses (cost 2)
  S3 -> S4 : Verify Fee Status (cost 1)
  S4 -> S5 : Confirm Enrollment (cost 1)

Total Registration Steps (Cost) = 1 + 2 + 2 + 1 + 1 = 7
